# IDSIA nano-drone benchmark

The math is in [`../exploiting_symmetry.md`](../exploiting_symmetry.md). This notebook is the IDSIA split only: Square / Random / Chirp runs 1–3 train, run 4 validates, Melon tests.

Our GRUs are already trained. Load them from disk. Then load Busetto et al.'s physics and physics+residual baselines from their clone under `nanodrone-sysid-benchmark/` (train those two only if the checkpoints are missing). Visualizations are next, not here.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "quadrotor" / "idsia").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quadrotor.evaluate import benchmark_mae
from quadrotor.idsia.baselines import load_or_train_idsia_baselines
from quadrotor.idsia.data import DEFAULT_ROOT, load_idsia_prepared
from quadrotor.idsia.models import load_trained

DEVICE = "cpu"
print("repo", ROOT)
print("IDSIA clone", DEFAULT_ROOT)
print("device", DEVICE)

In [ ]:
idsia_train, idsia_val, idsia_test = load_idsia_prepared()
print(
    f"train {len(idsia_train)}  val {len(idsia_val)}  test {len(idsia_test)}  "
    f"names {[flight.name for flight in idsia_test]}"
)

In [ ]:
ours = load_trained(device=DEVICE)
for name, model in ours.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"loaded {name:10s}  params {n_params}")

In [ ]:
baselines = load_or_train_idsia_baselines(
    idsia_train,
    idsia_val,
    device=DEVICE,
    train_if_missing=True,
)
for name, model in baselines.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"baseline {name:10s}  params {n_params}")

In [ ]:
def mae_row(name, model, flights):
    metrics = benchmark_mae(model, flights, start_stride=10)
    pos = metrics["position"]
    return {
        "model": name,
        "h=1 cm": 100 * float(pos[0]),
        "h=10 cm": 100 * float(pos[9]),
        "h=50 cm": 100 * float(pos[49]),
    }

rows = []
for name, model in {**ours, **baselines}.items():
    rows.append(mae_row(name, model, idsia_test))
    print(rows[-1])